# ============================================================
# CELL 1 — Imports & Configuration
# ============================================================

In [ ]:
import asyncio
import os
import re
import random
import urllib.parse
import warnings
from datetime import datetime
from urllib.parse import urlparse, parse_qs, unquote

import nest_asyncio
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

nest_asyncio.apply()
warnings.filterwarnings("ignore")

# ── User config ───────────────────────────────────────────────────────────────
SEARCH_TERMS = ["at home care near", "care homes near"]
DELAY_MIN    = 5          # seconds between requests (keep ≥5 to avoid blocks)
DELAY_MAX    = 11
# Edit this path to wherever you want the CSV saved
RESULTS_FILE = os.path.expanduser("~/durham_care_scrape_7.csv")
HEADLESS     = True       # False = watch the browser (useful for debugging)

# ── BrightData Scraping Browser config ───────────────────────────────────────
# 1. Sign up at https://brightdata.com
# 2. Create a zone: Dashboard → Add zone → Scraping Browser
# 3. Note your Customer ID, zone name, and zone password below
# 4. No SSL certificate needed — BrightData's browser runs on their servers
#
# Leave BRD_CUSTOMER_ID = "" to run WITHOUT proxy (local IP, ads contaminated).
BRD_CUSTOMER_ID = "hl_300427ba"                # e.g. "hl_abc12345"  — from your dashboard
BRD_ZONE        = "scraping_browser"  # your Scraping Browser zone name
BRD_PASSWORD    = "XXXXXXXXX"                # zone password from dashboard

# Postcode district → city coordinates for Proxy.setLocation CDP command
# BrightData geo-targets by lat/lon, not city name, in Scraping Browser
DISTRICT_TO_COORDS = {
    "DH": {"latitude": 54.7753, "longitude": -1.5849},  # Durham
    "DL": {"latitude": 54.5236, "longitude": -1.5589},  # Darlington
    "SR": {"latitude": 54.9069, "longitude": -1.3838},  # Sunderland
    "TS": {"latitude": 54.5742, "longitude": -1.2350},  # Middlesbrough
}

def brd_wss_url() -> str | None:
    """Return the BrightData Scraping Browser websocket URL, or None if unconfigured."""
    if not BRD_CUSTOMER_ID or not BRD_PASSWORD:
        return None
    return (
        f"wss://brd-customer-{BRD_CUSTOMER_ID}-zone-{BRD_ZONE}"
        f"-country-gb:{BRD_PASSWORD}@brd.superproxy.io:9222"
    )

# Care-related keyword filter for organic results
CARE_RE = re.compile(
    r"\b(care|caring|homecare|carers?|nursing|health|support|assist|"
    r"live.?in|domiciliary|elderly|dementia|residential|lodge|manor|"
    r"house|court|grange|hall|place|home)\b",
    re.IGNORECASE,
)

print("✅ Cell 1 complete — imports loaded")
print(f"   Results will save to: {RESULTS_FILE}")


✅ Cell 1 complete — imports loaded
   Results will save to: /Users/wolf6040/durham_care_scrape_7.csv


# ============================================================
# CELL 2 — Durham Postcode Sectors
# ============================================================

In [5]:
DURHAM_DISTRICTS = {
    "DH1":  range(1, 6),
    "DH2":  range(1, 4),
    "DH3":  range(1, 5),
    "DH4":  range(1, 7),
    "DH5":  range(1, 9),
    "DH6":  range(1, 6),
    "DH7":  range(1, 9),
    "DH8":  range(1, 8),
    "DH9":  range(1, 8),
    "DL4":  range(1, 3),
    "DL14": range(1, 9),
    "DL15": range(1, 6),
    "DL16": range(1, 7),
    "DL17": range(1, 7),
    "SR7":  range(1, 6),
    "SR8":  range(1, 6),
    "TS28": range(1, 3),
    "TS29": range(1, 3),
}

postcode_sectors = [
    f"{district} {sector}"
    for district, sectors in DURHAM_DISTRICTS.items()
    for sector in sectors
]

print(f"✅ Cell 2 complete")
print(f"   Postcode sectors : {len(postcode_sectors)}")
print(f"   Total queries    : {len(postcode_sectors) * len(SEARCH_TERMS)}")
print(f"   Sample           : {postcode_sectors[:8]}")


✅ Cell 2 complete
   Postcode sectors : 94
   Total queries    : 188
   Sample           : ['DH1 1', 'DH1 2', 'DH1 3', 'DH1 4', 'DH1 5', 'DH2 1', 'DH2 2', 'DH2 3']


# ── Cell 2b — Fetch one real postcode per sector via postcodes.io ──

In [6]:
import aiohttp

async def get_postcode_for_sector(session, sector):
    url = f"https://api.postcodes.io/postcodes?q={urllib.parse.quote(sector)}&limit=1"
    headers = {"Accept-Encoding": "gzip, deflate"}
    try:
        async with session.get(url, headers=headers, timeout=aiohttp.ClientTimeout(total=10)) as resp:
            data = await resp.json(content_type=None)
            if data["status"] == 200 and data["result"]:
                return sector, data["result"][0]["postcode"]
    except Exception as e:
        print(f"  ⚠ {sector}: {e}")
    return sector, None

async def fetch_all_sector_postcodes(sectors):
    async with aiohttp.ClientSession() as session:
        tasks = [get_postcode_for_sector(session, s) for s in sectors]
        results = await asyncio.gather(*tasks)
    return dict(results)

sector_to_postcode = asyncio.run(fetch_all_sector_postcodes(postcode_sectors))

missing = [s for s, pc in sector_to_postcode.items() if pc is None]
if missing:
    print(f"⚠ No postcode found for {len(missing)} sectors: {missing}")
else:
    print(f"✅ Got full postcodes for all {len(sector_to_postcode)} sectors")

for sector, pc in list(sector_to_postcode.items())[:6]:
    print(f"  {sector}  →  {pc}")


⚠ No postcode found for 51 sectors: ['DH4 1', 'DH4 2', 'DH4 3', 'DH5 1', 'DH5 2', 'DH5 3', 'DH5 4', 'DH5 5', 'DH5 6', 'DH5 7', 'DH7 1', 'DH7 2', 'DH7 3', 'DH7 4', 'DH7 5', 'DH8 2', 'DH8 3', 'DH8 4', 'DH9 1', 'DH9 2', 'DH9 3', 'DH9 4', 'DH9 5', 'DL14 1', 'DL14 2', 'DL14 3', 'DL14 5', 'DL15 1', 'DL15 2', 'DL15 3', 'DL15 4', 'DL15 5', 'DL16 1', 'DL16 2', 'DL16 3', 'DL16 4', 'DL16 5', 'DL17 1', 'DL17 2', 'DL17 3', 'DL17 4', 'DL17 5', 'DL17 6', 'SR7 2', 'SR7 3', 'SR7 4', 'SR7 5', 'TS28 1', 'TS28 2', 'TS29 1', 'TS29 2']
  DH1 1  →  DH1 1AB
  DH1 2  →  DH1 2AA
  DH1 3  →  DH1 3AA
  DH1 4  →  DH1 4AA
  DH1 5  →  DH1 5AA
  DH2 1  →  DH2 1AB


# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CELL 3 — HTML Parser + Ad URL Extraction                                    │
# └─────────────────────────────────────────────────────────────────────────────┘

In [7]:
# ============================================================
# CELL 3 — HTML Parser
# Classifies: Ad | Local Pack | Organic
# Ads also capture ad_destination_url (actual provider website)
# extracted from the adurl= param in the /aclk? tracking href.
# This enables CQC matching via URL.
# ============================================================

DEBUG_MODE      = False
DEBUG_HTML_PATH = os.path.expanduser("~/diag_real_serp.html")
LOCAL_PACK_TITLES = {"map", "more places"}
_AD_SKIP = {"website", "directions", "map", "more places", ""}


def _parse_adurl(aclk_href: str) -> str:
    """Extract the real landing-page URL from a Google /aclk? tracking href.

    Google encodes the destination in the adurl= query parameter.
    Returns the decoded URL, or "" if absent / not an http URL.
    This is the URL to use for CQC provider matching.
    """
    try:
        qs = parse_qs(urlparse(aclk_href).query)
        adurl = qs.get("adurl", [""])[0]
        if adurl and adurl.startswith("http"):
            return unquote(adurl)
    except Exception:
        pass
    return ""


def extract_provider_name(title: str) -> str:
    title = re.sub(r"\s*[-–|·].*$", "", title).strip()
    title = re.sub(r"https?://\S+", "", title).strip()
    return title if title else "Unknown"


def get_reported_location(soup, postcode_sector: str) -> str:
    """Target span.gm7Ysb — Google's 'Results for' label."""
    for span in soup.find_all("span", class_="gm7Ysb"):
        parent = span.find_parent()
        if not parent:
            continue
        full = parent.get_text(separator=" ", strip=True)
        stripped = re.sub(r"Results\s+for[:\s]*", "", full, flags=re.IGNORECASE).strip()
        if stripped and len(stripped) > 2:
            return stripped
    return postcode_sector


def extract_ads(soup, postcode_sector, full_postcode, search_term, reported_location):
    """Extract paid text ads.

    Captures both the /aclk? tracking URL (url) and the decoded
    destination URL (ad_destination_url) for provider identification.
    Returns (list of ad rows, set of ad titles seen).
    """
    ad_titles_seen = set()
    rows = []
    for a_tag in soup.find_all("a", href=re.compile(r"/aclk\?")):
        title = a_tag.get_text(strip=True)
        if title.lower() in _AD_SKIP or title in ad_titles_seen:
            continue
        ad_titles_seen.add(title)
        aclk_href = a_tag["href"]
        rows.append({
            "postcode_sector":    postcode_sector,
            "full_postcode":      full_postcode,
            "search_term":        search_term,
            "result_type":        "Ad",
            "position":           len(rows) + 1,
            "provider_name":      extract_provider_name(title),
            "title":              title,
            "url":                aclk_href,          # Google tracking URL
            "ad_destination_url": _parse_adurl(aclk_href),  # actual provider URL
            "reported_location":  reported_location,
            "scraped_at":         datetime.utcnow().isoformat(),
        })
    return rows, ad_titles_seen


def probe_html(soup, postcode_sector):
    """Print diagnostic probes to help identify DOM elements."""
    print("\n" + "="*65)
    print("PROBE 1 — Location hint text nodes")
    print("="*65)
    loc_re = re.compile(r"(Results\s+for|DH\d|Durham|County Durham|near|postcode)", re.I)
    for node in soup.find_all(string=loc_re):
        text = node.strip()
        if not text or len(text) > 300:
            continue
        parent = node.find_parent()
        tag    = parent.name if parent else "?"
        cls    = " ".join(parent.get("class", [])) if parent else ""
        print(f"\n  TEXT  : {repr(text[:120])}")
        print(f"  TAG   : <{tag}> class='{cls}'")

    print("\n" + "="*65)
    print("PROBE 2 — /aclk links & their adurl= destination")
    print("="*65)
    aclk = soup.find_all("a", href=re.compile(r"/aclk\?"))
    print(f"  /aclk links found: {len(aclk)}")
    for a in aclk[:8]:
        link_text = a.get_text(strip=True)[:80]
        dest_url  = _parse_adurl(a["href"])
        print(f"\n  link text    : {repr(link_text)}")
        print(f"  dest url     : {dest_url[:100]}")

    print("\n" + "="*65)
    print("PROBE 3 — All h3 titles on page")
    print("="*65)
    for i, h3 in enumerate(soup.find_all("h3"), 1):
        t = h3.get_text(strip=True)
        if t:
            print(f"  [{i:02}] {t[:100]}")


def parse_rendered_html(
    html: str,
    postcode_sector: str,
    full_postcode: str,
    search_term: str,
) -> list[dict]:
    """Parse a fully-rendered Google SERP into structured rows.

    Classifies results as Ad | Local Pack | Organic.
    Ad rows include ad_destination_url (real provider website) in addition
    to the /aclk? tracking URL — use ad_destination_url for CQC matching.
    """
    soup = BeautifulSoup(html, "html.parser")
    results = []
    reported_location = get_reported_location(soup, postcode_sector)

    if DEBUG_MODE:
        print(f"\n🔍 DEBUG probe for: {search_term} {postcode_sector}")
        print(f"   HTML length       : {len(html):,}")
        print(f"   reported_location : {repr(reported_location)}")
        probe_html(soup, postcode_sector)

    # ── Paid text ads ─────────────────────────────────────────────────────────
    ad_rows, ad_titles_seen = extract_ads(
        soup, postcode_sector, full_postcode, search_term, reported_location
    )
    results.extend(ad_rows)

    # ── Local Pack ────────────────────────────────────────────────────────────
    local_seen = set()
    for node in soup.find_all(string=re.compile(r"^(Map|More places)$", re.I)):
        label = node.strip()
        if label in local_seen:
            continue
        local_seen.add(label)
        parent = node.find_parent("a") or node.find_parent("div")
        url = (parent.get("href", "") if parent and parent.name == "a" else "")
        results.append({
            "postcode_sector":    postcode_sector,
            "full_postcode":      full_postcode,
            "search_term":        search_term,
            "result_type":        "Local Pack",
            "position":           1,
            "provider_name":      "Google Local Pack",
            "title":              label,
            "url":                url,
            "ad_destination_url": "",
            "reported_location":  reported_location,
            "scraped_at":         datetime.utcnow().isoformat(),
        })

    # ── Organic results ───────────────────────────────────────────────────────
    organic_seen = set(ad_titles_seen)
    for h3 in soup.find_all("h3"):
        title = h3.get_text(strip=True)
        if (
            not title
            or title in organic_seen
            or title.lower() in LOCAL_PACK_TITLES
            or not CARE_RE.search(title)
        ):
            continue
        container = h3.find_parent("div")
        url = ""
        for _ in range(8):
            if not container:
                break
            link = container.find("a", href=True)
            if link and str(link["href"]).startswith("http"):
                url = link["href"]
                break
            container = container.find_parent("div")
        organic_seen.add(title)
        results.append({
            "postcode_sector":    postcode_sector,
            "full_postcode":      full_postcode,
            "search_term":        search_term,
            "result_type":        "Organic",
            "position":           sum(1 for r in results if r["result_type"] == "Organic") + 1,
            "provider_name":      extract_provider_name(title),
            "title":              title,
            "url":                url,
            "ad_destination_url": "",
            "reported_location":  reported_location,
            "scraped_at":         datetime.utcnow().isoformat(),
        })

    return results


if DEBUG_MODE:
    if os.path.exists(DEBUG_HTML_PATH):
        print(f"Loading saved HTML from {DEBUG_HTML_PATH}")
        with open(DEBUG_HTML_PATH, encoding="utf-8") as f:
            _html = f.read()
        parse_rendered_html(_html, "DH1 1", "DH1 1AB", "care homes near")
    else:
        print(f"⚠ DEBUG_MODE=True but no file at {DEBUG_HTML_PATH}")
        print(f"  Run Cell 4 once with DEBUG_SAVE=True first.")
else:
    print("✅ Cell 3 complete — parser defined (ad_destination_url enabled)")


✅ Cell 3 complete — parser defined (ad_destination_url enabled)


# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CELL 4 — Playwright Scrape Engine                                           │
# └─────────────────────────────────────────────────────────────────────────────┘

In [8]:
# ============================================================
# CELL 4 — Playwright Scrape Engine (BrightData Scraping Browser)
# Must run AFTER Cell 3 (parser) and BEFORE Cell 5 (runner)
#
# If BRD_CUSTOMER_ID is set in Cell 1, connects to BrightData's
# remote Scraping Browser over websocket — no local browser launch,
# no SSL cert needed. City-level geo-targeting is set per query
# via the Proxy.setLocation CDP command.
#
# If BRD_CUSTOMER_ID is empty, falls back to launching a local
# Chromium browser (Oxford IP — ads will be location-contaminated).
#
# Resource blocking cuts ~60% bandwidth by aborting images,
# fonts, media, and analytics requests.
# ============================================================

DEBUG_SAVE      = True
DEBUG_SAVE_PATH = os.path.expanduser("~/diag_real_serp.html")

_GOOGLE_BASE    = "https://www.google.co.uk"
_AD_REDIRECT_TIMEOUT = 8000
_NAV_TIMEOUT    = 60_000   # ms — BrightData remote browser can be slower

# Resource types and domains to block (saves bandwidth)
_BLOCK_TYPES   = {"image", "media", "font", "other"}
_BLOCK_DOMAINS = {
    "google-analytics.com", "googletagmanager.com", "doubleclick.net",
    "googlesyndication.com", "analytics.google.com",
}


async def _block_unnecessary(route, request):
    if request.resource_type in _BLOCK_TYPES:
        await route.abort()
        return
    if any(d in request.url for d in _BLOCK_DOMAINS):
        await route.abort()
        return
    await route.continue_()


async def _set_location(page, district: str):
    """Set Proxy.setLocation via CDP so BrightData routes through the right city."""
    coords = DISTRICT_TO_COORDS.get(district[:2], DISTRICT_TO_COORDS["DH"])
    try:
        client = await page.context.new_cdp_session(page)
        await client.send("Proxy.setLocation", {
            "latitude":  coords["latitude"],
            "longitude": coords["longitude"],
            "distance":  10,     # km radius
            "strict":    False,  # expand if no peer found nearby
        })
    except Exception as e:
        print(f"  ⚠ Proxy.setLocation failed (non-fatal): {e}")


async def resolve_ad_destination(browser, aclk_href: str) -> str:
    """Follow an /aclk? redirect in a background tab and return the final URL."""
    full_url = _GOOGLE_BASE + aclk_href if aclk_href.startswith("/") else aclk_href
    tab = await browser.new_page()
    try:
        try:
            await tab.route("**/*", _block_unnecessary)
        except Exception:
            pass  # remote CDP browsers don't support route interception
        await tab.goto(full_url, wait_until="commit", timeout=_AD_REDIRECT_TIMEOUT)
        await asyncio.sleep(1.5)
        final_url = tab.url
        if "google." in final_url:
            return ""
        return final_url
    except Exception:
        return ""
    finally:
        await tab.close()


async def scrape_all(
    queries: list[tuple[str, str, str]],
    headless: bool = True,   # only used for local fallback
) -> list[dict]:
    """Scrape Google SERPs for each (full_postcode, sector, search_term) tuple."""
    all_results = []
    total = len(queries)
    debug_saved = False
    wss_url = brd_wss_url()

    async with async_playwright() as pw:

        # ── Connect to BrightData Scraping Browser OR local Chromium ──────────
        if wss_url:
            print("🌐 Connecting to BrightData Scraping Browser…")
            browser = await pw.chromium.connect_over_cdp(wss_url, timeout=30_000)
            print("   Connected! (geo-targeting via Proxy.setLocation per query)")
            # connect_over_cdp returns a browser with a pre-existing default context —
            # must use that context rather than calling browser.new_page() directly
            context = browser.contexts[0]
            page = await context.new_page()
        else:
            print("⚠️  No BrightData config — launching local browser")
            print("   Ads will reflect your local IP location, not search area.")
            browser = await pw.chromium.launch(
                headless=headless,
                args=[
                    "--no-sandbox",
                    "--disable-blink-features=AutomationControlled",
                    "--disable-dev-shm-usage",
                ],
            )
            page = await browser.new_page()
            # Local browser: spoof navigator properties
            await page.add_init_script("""
                Object.defineProperty(navigator, 'webdriver',  {get: () => undefined});
                Object.defineProperty(navigator, 'languages',  {get: () => ['en-GB','en']});
                Object.defineProperty(navigator, 'plugins',    {get: () => [1, 2, 3]});
            """)

        page.set_default_navigation_timeout(_NAV_TIMEOUT)

        # ── Resource blocking ─────────────────────────────────────────────────
        # page.route() doesn't work on remote CDP browsers (BrightData).
        # Use their native CDP Unblocker.enableAdBlock instead; fall back to
        # page.route() for local Chromium.
        if wss_url:
            try:
                client = await page.context.new_cdp_session(page)
                await client.send("Unblocker.enableAdBlock")
            except Exception:
                pass  # non-fatal — older zones may not support this
        else:
            await page.route("**/*", _block_unnecessary)

        # ── Consent banner ────────────────────────────────────────────────────
        print("Handling Google consent banner…")
        await page.goto("https://www.google.co.uk", wait_until="domcontentloaded",
                        timeout=_NAV_TIMEOUT)
        await asyncio.sleep(2)
        for btn_text in ["Accept all", "Reject all", "I agree"]:
            try:
                btn = page.get_by_role("button", name=btn_text)
                if await btn.count() > 0:
                    await btn.first.click()
                    print(f"  ✓ Consent: '{btn_text}'")
                    await asyncio.sleep(2)
                    break
            except Exception:
                pass

        current_district = None

        for i, (full_postcode, sector, term) in enumerate(queries, 1):
            district = sector.split()[0][:2]   # e.g. "DH" from "DH1 1"

            # ── Set geo-location when district changes ─────────────────────
            if wss_url and district != current_district:
                await _set_location(page, district)
                coords = DISTRICT_TO_COORDS.get(district, DISTRICT_TO_COORDS["DH"])
                print(f"\n📍 Location set → {district} "
                      f"({coords['latitude']:.4f}, {coords['longitude']:.4f})")
                current_district = district

            query      = f"{term} {full_postcode}"
            search_url = (
                f"https://www.google.co.uk/search"
                f"?q={urllib.parse.quote(query)}&hl=en-GB&gl=GB&num=20"
            )
            print(f"[{i}/{total}] {query}", end=" … ")

            try:
                await page.goto(search_url, wait_until="networkidle",
                                timeout=_NAV_TIMEOUT)
                await asyncio.sleep(random.uniform(1.5, 3.0))

                body_text = (await page.inner_text("body")).lower()
                if "unusual traffic" in body_text or "captcha" in body_text:
                    print("🚨 CAPTCHA detected — waiting 60s…")
                    await asyncio.sleep(60)
                    await page.goto(search_url, wait_until="networkidle",
                                    timeout=_NAV_TIMEOUT)
                    await asyncio.sleep(4)

                html = await page.content()

                if DEBUG_SAVE and not debug_saved:
                    with open(DEBUG_SAVE_PATH, "w", encoding="utf-8") as f:
                        f.write(html)
                    print(f"\n  💾 Debug HTML saved → {DEBUG_SAVE_PATH}")
                    debug_saved = True

                rows = parse_rendered_html(html, sector, full_postcode, term)

                # ── Resolve missing ad destination URLs ───────────────────
                unresolved = [
                    r for r in rows
                    if r["result_type"] == "Ad" and not r["ad_destination_url"]
                ]
                if unresolved:
                    print(f"\n  🔗 Resolving {len(unresolved)} ad redirect(s)…", end=" ")
                    for r in unresolved:
                        dest = await resolve_ad_destination(browser, r["url"])
                        r["ad_destination_url"] = dest
                    resolved_n = sum(1 for r in unresolved if r["ad_destination_url"])
                    print(f"{resolved_n}/{len(unresolved)} resolved")

                all_results.extend(rows)

                ad_n   = sum(1 for r in rows if r["result_type"] == "Ad")
                org_n  = sum(1 for r in rows if r["result_type"] == "Organic")
                lp_n   = sum(1 for r in rows if r["result_type"] == "Local Pack")
                dest_n = sum(1 for r in rows if r.get("ad_destination_url"))
                print(f"✓  {len(rows)} results  "
                      f"[Ads:{ad_n}(dest:{dest_n})  Organic:{org_n}  LocalPack:{lp_n}]")

            except Exception as e:
                print(f"✗  {e}")

            await asyncio.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

        await browser.close()

    return all_results


proxy_status = "enabled" if brd_wss_url() else "NOT configured (local IP fallback)"
print(f"✅ Cell 4 ready — scrape_all defined")
print(f"   BrightData Scraping Browser : {proxy_status}")
print(f"   Resource blocking           : images, fonts, media, analytics blocked")
print(f"   Navigation timeout          : {_NAV_TIMEOUT//1000}s per page")
print(f"   DEBUG_SAVE                  : {DEBUG_SAVE}  →  {DEBUG_SAVE_PATH}")


✅ Cell 4 ready — scrape_all defined
   BrightData Scraping Browser : enabled
   Resource blocking           : images, fonts, media, analytics blocked
   Navigation timeout          : 60s per page
   DEBUG_SAVE                  : True  →  /Users/wolf6040/diag_real_serp.html


# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CELL 5 — Run the Scrape  (saves CSV — supports resume)                      │
# │          ⚠ Only run this cell when you want to (re)scrape                   │
# └─────────────────────────────────────────────────────────────────────────────┘

In [24]:
# ============================================================
# CELL 5 — Run the Scrape  (saves CSV — supports resume)
# ⚠  Only run this cell when you want to (re)scrape
# ============================================================

if os.path.exists(RESULTS_FILE):
    df_existing = pd.read_csv(RESULTS_FILE)
    done_keys = set(df_existing["postcode_sector"] + "||" + df_existing["search_term"])
    print(f"Resuming — {len(df_existing):,} rows already in {RESULTS_FILE}")
else:
    df_existing = pd.DataFrame()
    done_keys   = set()
    print("Starting fresh scrape")

todo = [
    (sector_to_postcode[pc], pc, term)   # (full_postcode, sector, term)
    for pc   in postcode_sectors
    for term in SEARCH_TERMS
    if f"{pc}||{term}" not in done_keys
    and sector_to_postcode.get(pc)
]
print(f"Queries remaining: {len(todo)}")

if todo:
    new_results = asyncio.run(scrape_all(todo, headless=HEADLESS))
    df_new      = pd.DataFrame(new_results)
    df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    df_combined.to_csv(RESULTS_FILE, index=False)
    print(f"\n✅  Saved {len(df_combined):,} total rows → {RESULTS_FILE}")
else:
    print("Nothing to scrape — all queries already complete.")


Starting fresh scrape
Queries remaining: 86
🌐 Connecting to BrightData Scraping Browser…
   Connected! (geo-targeting via Proxy.setLocation per query)
Handling Google consent banner…
  ⚠ Proxy.setLocation failed (non-fatal): CDPSession.send: Protocol error (Proxy.setLocation): Cannot set proxy location after page navigated (set_location_after_navigate)

📍 Location set → DH (54.7753, -1.5849)
[1/86] at home care near DH1 1AB … 
  💾 Debug HTML saved → /Users/wolf6040/diag_real_serp.html

  🔗 Resolving 2 ad redirect(s)… 2/2 resolved
✓  15 results  [Ads:4(dest:4)  Organic:9  LocalPack:2]
[2/86] care homes near DH1 1AB … 🚨 CAPTCHA detected — waiting 60s…
✓  0 results  [Ads:0(dest:0)  Organic:0  LocalPack:0]
[3/86] at home care near DH1 2AA … 🚨 CAPTCHA detected — waiting 60s…
✓  0 results  [Ads:0(dest:0)  Organic:0  LocalPack:0]
[4/86] care homes near DH1 2AA … 🚨 CAPTCHA detected — waiting 60s…
✓  0 results  [Ads:0(dest:0)  Organic:0  LocalPack:0]
[5/86] at home care near DH1 3AA … 🚨 CAPTCHA

# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CELL 5b — Back-fill ad destination URLs for existing CSV                    │
# │           Run once after scraping to resolve missing ad URLs                │
# └─────────────────────────────────────────────────────────────────────────────┘

In [10]:
# ============================================================
# CELL 5b — Back-fill ad_destination_url for existing CSV
# Run this ONCE to resolve the 79% of ads missing a dest URL.
# Uses Playwright to follow each /aclk? redirect in a tab.
# No re-scraping of SERPs needed.
# ============================================================

async def backfill_ad_destinations(csv_path: str, headless: bool = True):
    df_bf = pd.read_csv(csv_path)

    # Identify rows that need resolving
    mask = (
        (df_bf["result_type"] == "Ad") &
        (df_bf["url"].str.contains("/aclk?", regex=False, na=False)) &
        (df_bf["ad_destination_url"].fillna("").str.len() == 0)
    )
    todo_idx = df_bf[mask].index.tolist()
    print(f"Rows needing resolution : {len(todo_idx)}")
    if not todo_idx:
        print("Nothing to do — all ad destination URLs already populated.")
        return df_bf

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(
            headless=headless,
            args=["--no-sandbox", "--disable-blink-features=AutomationControlled"],
        )
        context = await browser.new_context(
            viewport={"width": 1280, "height": 900},
            locale="en-GB",
            timezone_id="Europe/London",
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            ),
        )

        resolved = 0
        for n, idx in enumerate(todo_idx, 1):
            aclk_href = df_bf.at[idx, "url"]
            dest = await resolve_ad_destination(context, aclk_href)
            df_bf.at[idx, "ad_destination_url"] = dest
            if dest:
                resolved += 1
            if n % 20 == 0 or n == len(todo_idx):
                print(f"  [{n}/{len(todo_idx)}] resolved so far: {resolved}")
            # Small delay to avoid hammering Google
            await asyncio.sleep(random.uniform(1.5, 3.0))

        await browser.close()

    df_bf.to_csv(csv_path, index=False)
    pct = resolved / len(todo_idx) * 100
    print(f"\n✅ Back-fill complete: {resolved}/{len(todo_idx)} ({pct:.0f}%) resolved")
    print(f"   CSV saved → {csv_path}")
    return df_bf


df = asyncio.run(backfill_ad_destinations(RESULTS_FILE, headless=HEADLESS))

# Reload and re-run the Cell 6 cleaning so df is up to date in memory
ads = df[df["result_type"] == "Ad"].copy()
has_dest = ads["ad_destination_url"].fillna("").str.len() > 0
print(f"\nAd rows with destination URL now: {has_dest.sum()} / {len(ads)}  ({has_dest.mean()*100:.0f}%)")


Rows needing resolution : 0
Nothing to do — all ad destination URLs already populated.

Ad rows with destination URL now: 117 / 117  (100%)


# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CELL 6 — Load, Clean & Classify  (start here on re-runs)                   │
# └─────────────────────────────────────────────────────────────────────────────┘

In [9]:
# ============================================================
# CELL 6 — Load, Clean & Classify  (start here on re-runs)
# ============================================================

df = pd.read_csv(RESULTS_FILE)

# ── Repair / populate ad_destination_url ─────────────────────────────────────
# Always re-derive from the raw /aclk? URL so that:
#   (a) old CSVs that lacked the column get it back-filled, and
#   (b) newer CSVs where empty strings were read back as NaN get fixed.
def _fill_ad_dest(row):
    if str(row.get("result_type", "")) == "Ad":
        return _parse_adurl(str(row.get("url", "")))
    return ""

df["ad_destination_url"] = df.apply(_fill_ad_dest, axis=1)

# ── Reclassify result_type ────────────────────────────────────────────────────
LOCAL_PACK_TITLES = {"map", "more places"}

def reclassify(row):
    url   = str(row.get("url", ""))
    title = str(row.get("title", "")).strip().lower()
    if title in LOCAL_PACK_TITLES:
        return "Local Pack"
    if "/aclk?" in url or url.startswith("/aclk"):
        return "Ad"
    return "Organic"

df["result_type"] = df.apply(reclassify, axis=1)

# ── Fix reported_location ─────────────────────────────────────────────────────
def clean_location(row):
    loc  = str(row.get("reported_location", "")).strip()
    bare = re.sub(r"^results\s*for[:\s]*", "", loc, flags=re.IGNORECASE).strip()
    if not bare or bare.lower() in {"results for", "nan", ""}:
        return row["postcode_sector"]
    return bare

df["reported_location"] = df.apply(clean_location, axis=1)

# ── Standard field cleaning ───────────────────────────────────────────────────
df["position"]    = pd.to_numeric(df["position"], errors="coerce")
df["scraped_at"]  = pd.to_datetime(df["scraped_at"], errors="coerce")
df["district"]    = df["postcode_sector"].str.extract(r"^([A-Z]+\d+)", expand=False)
df["search_type"] = df["search_term"].apply(
    lambda x: "Home Care" if "at home" in str(x).lower() else "Care Homes"
)

# ── Report ────────────────────────────────────────────────────────────────────
print(f"Rows            : {len(df):,}")
print(f"Sectors covered : {df['postcode_sector'].nunique()}")
print(f"\nResult type counts:")
print(df["result_type"].value_counts().to_string())

ads = df[df["result_type"] == "Ad"].copy()
has_dest = ads["ad_destination_url"].fillna("").str.len() > 0
print(f"\nAd rows with destination URL : {has_dest.sum()} / {len(ads)}  ({has_dest.mean()*100:.0f}%)")
print(f"\nSample ad_destination_url values (non-empty):")
sample = ads.loc[has_dest, "ad_destination_url"].head(5)
for v in sample:
    print(f"  {v[:80]}")
print(f"\nNote: ads with empty ad_destination_url had no adurl= param in the /aclk? href.")
print(f"      These require a re-scrape with Cell 5 to capture.")


Rows            : 709
Sectors covered : 29

Result type counts:
result_type
Organic       494
Ad            117
Local Pack     98

Ad rows with destination URL : 51 / 117  (44%)

Sample ad_destination_url values (non-empty):
  https://comparecaring.com/home-care/?gad_source=1&gad_campaignid=22910915203&gbr
  https://comparecaring.com/home-care/?gad_source=1&gad_campaignid=22910915203&gbr
  https://comparecaring.com/home-domiciliary-care/?gad_source=1&gad_campaignid=229
  https://comparecaring.com/home-domiciliary-care/?gad_source=1&gad_campaignid=229
  https://www.allcarehomes.co.uk/home-mobile?&campaignid=23517043335&adgroupid=191

Note: ads with empty ad_destination_url had no adurl= param in the /aclk? href.
      These require a re-scrape with Cell 5 to capture.


# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CELL 6b — URL Domain Extraction for CQC Matching                           │
# └─────────────────────────────────────────────────────────────────────────────┘

In [10]:
# ============================================================
# CELL 6b — URL Domain Extraction for CQC Matching
# ============================================================
# pip install tldextract  (if not already installed)
#
# Creates two new columns:
#   effective_url   — the best URL for this row (ad_destination_url
#                     for Ads, url for Organic, blank for Local Pack)
#   url_domain      — the registrable domain core, e.g.:
#                       https://get.oxfordaunts.co.uk/... → "oxfordaunts"
#                       https://www.homecare.co.uk/...    → "homecare"
#                       https://care.country-cousins.co.uk → "country-cousins"
#                       https://www.elder.org/...          → "elder"
#                     This is what to join against CQC's equivalent field.
# ============================================================

try:
    import tldextract
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tldextract", "-q"])
    import tldextract

import urllib.parse


def get_effective_url(row) -> str:
    """Pick the most useful URL for this row."""
    rt = row.get("result_type", "")
    if rt == "Ad":
        dest = str(row.get("ad_destination_url", "")).strip()
        return dest if dest and dest != "nan" else ""
    elif rt == "Organic":
        u = str(row.get("url", "")).strip()
        return u if u and u != "nan" else ""
    return ""  # Local Pack


def extract_domain_core(url: str) -> str:
    """Return just the registrable domain name, without subdomain or TLD.

    Uses tldextract to correctly handle multi-part TLDs like .co.uk, .org.uk.
    Also strips trailing tracking junk from the URL before parsing.

    Examples:
        https://get.oxfordaunts.co.uk/home-care-service/?tracking=... → oxfordaunts
        https://www.homecare.co.uk/homecare/listings.cfm              → homecare
        https://care.country-cousins.co.uk/elderly-care/             → country-cousins
        https://www.elder.org/why-elder/                              → elder
        https://www.homeinstead.co.uk/durham/                         → homeinstead
    """
    if not url or not url.startswith("http"):
        return ""
    # Strip query string and fragment before extracting — avoids tldextract
    # being confused by URLs that embed other URLs in tracking params
    try:
        parsed = urllib.parse.urlparse(url)
        clean  = urllib.parse.urlunparse(parsed._replace(query="", fragment=""))
    except Exception:
        clean = url
    ext = tldextract.extract(clean)
    return ext.domain  # e.g. "oxfordaunts", "homecare", "country-cousins"


# ── Apply ─────────────────────────────────────────────────────────────────────
df["effective_url"] = df.apply(get_effective_url, axis=1)
df["url_domain"]    = df["effective_url"].apply(extract_domain_core)

# ── Report ────────────────────────────────────────────────────────────────────
print("Sample URL → domain mapping:")
sample = (
    df[df["effective_url"].str.len() > 0]
    [["result_type", "effective_url", "url_domain"]]
    .drop_duplicates("url_domain")
    .head(15)
)
for _, row in sample.iterrows():
    print(f"  [{row.result_type:10}]  {row.effective_url[:55]:<55}  →  {row.url_domain}")

print(f"\nUnique domain cores  : {df['url_domain'].replace('', pd.NA).nunique()}")
print(f"Rows with domain     : {(df['url_domain'].str.len() > 0).sum()}")
print(f"Rows without domain  : {(df['url_domain'].str.len() == 0).sum()}  (Local Pack + unresolved ads)")

# ── CQC matching hint ─────────────────────────────────────────────────────────
print("\n── CQC join hint ─────────────────────────────────────────────────────────")
print("In your CQC data, apply the same extract_domain_core() to the website column,")
print("then merge on url_domain:")
print()
print("  cqc['url_domain'] = cqc['Website'].apply(extract_domain_core)")
print("  merged = df.merge(cqc, on='url_domain', how='left')")


Sample URL → domain mapping:
  [Ad        ]  https://comparecaring.com/home-care/?gad_source=1&gad_c  →  comparecaring
  [Organic   ]  https://www.homecare.co.uk/homecare/listings.cfm/search  →  homecare
  [Organic   ]  https://www.secondtononecare.co.uk/                      →  secondtononecare
  [Organic   ]  https://www.homeinstead.co.uk/durham/                    →  homeinstead
  [Organic   ]  https://countydurham.myhomecare.co.uk/                   →  myhomecare
  [Organic   ]  https://www.caresourcer.com/s/providers/towns/durham/ho  →  caresourcer
  [Organic   ]  https://www.blossomhomecare.co.uk/offices/durham/        →  blossomhomecare
  [Organic   ]  https://perfectcare.co.uk/                               →  perfectcare
  [Organic   ]  https://www.shinecareathome.co.uk/locations/home-care-i  →  shinecareathome
  [Organic   ]  https://www.carehome.co.uk/care_search_results.cfm/sear  →  carehome
  [Organic   ]  https://www.gainfordcarehomes.com/our-homes/newton-hall  →  gainfor

In [13]:
df['provider_name'] = df['url_domain']

df = df[df['provider_name']!=""]

# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CELL 7 — Summary Table  (Figure 1 — Table view)                            │
# └─────────────────────────────────────────────────────────────────────────────┘

In [11]:
# ============================================================
# CELL 7 — Summary Table (Figure 1 — Table view)
# ============================================================

def build_summary_table(df: pd.DataFrame) -> pd.DataFrame:
    """Provider-level summary: appearances split by Ad / Organic, mean position,
    sectors covered, districts.  Local Pack excluded (not individual providers)."""

    sub = df[df["result_type"].isin(["Ad", "Organic"])].copy()

    grp = (
        sub.groupby(["provider_name", "result_type", "search_type"])
        .agg(
            appearances   = ("postcode_sector", "count"),
            sectors_count = ("postcode_sector", "nunique"),
            mean_position = ("position", "mean"),
            districts     = ("district", lambda x: ", ".join(sorted(x.dropna().unique()))),
        )
        .reset_index()
    )

    pivot = grp.pivot_table(
        index=["provider_name", "search_type"],
        columns="result_type",
        values=["appearances", "mean_position"],
        aggfunc="first",
    ).fillna(0)

    pivot.columns = [f"{stat}_{rtype}" for stat, rtype in pivot.columns]
    pivot = pivot.reset_index()

    # Ensure all four columns exist even if one result_type absent in data
    for col in ["appearances_Ad", "appearances_Organic",
                "mean_position_Ad", "mean_position_Organic"]:
        if col not in pivot.columns:
            pivot[col] = 0

    # Force numeric — pivot_table with aggfunc='first' can return object dtype
    for col in ["appearances_Ad", "appearances_Organic",
                "mean_position_Ad", "mean_position_Organic"]:
        pivot[col] = pd.to_numeric(pivot[col], errors="coerce").fillna(0.0)

    pivot["total_appearances"] = pivot["appearances_Ad"] + pivot["appearances_Organic"]
    pivot = pivot.sort_values("total_appearances", ascending=False).reset_index(drop=True)
    pivot.insert(0, "rank", range(1, len(pivot) + 1))

    # Replace 0 with NaN (float, not pd.NA) so .round() keeps float64 dtype
    for col in ["mean_position_Ad", "mean_position_Organic"]:
        pivot[col] = pivot[col].where(pivot[col] != 0.0).round(1)

    pivot = pivot.rename(columns={
        "provider_name":         "Provider",
        "search_type":           "Search Type",
        "appearances_Ad":        "Ad Appearances",
        "appearances_Organic":   "Organic Appearances",
        "mean_position_Ad":      "Mean Ad Position",
        "mean_position_Organic": "Mean Organic Position",
        "total_appearances":     "Total Appearances",
    })

    return pivot


summary = build_summary_table(df)

styled = (
    summary.head(40).style
    .background_gradient(subset=["Total Appearances"],   cmap="YlOrRd")
    .background_gradient(subset=["Ad Appearances"],      cmap="Reds",  vmin=0)
    .background_gradient(subset=["Organic Appearances"], cmap="Blues", vmin=0)
    .format(
        {"Mean Ad Position": "{:.1f}", "Mean Organic Position": "{:.1f}"},
        na_rep="—",
    )
    .set_caption("Figure 1 — Table: Care Provider Visibility in Durham Google Search Results")
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "13px"), ("font-weight", "bold"),
                   ("text-align", "left"), ("padding-bottom", "8px")]},
        {"selector": "th",
         "props": [("background-color", "#2c3e50"), ("color", "white"),
                   ("font-size", "11px"), ("padding", "6px 10px")]},
        {"selector": "td",
         "props": [("font-size", "11px"), ("padding", "4px 10px")]},
    ])
)

display(styled)
summary.to_csv(os.path.expanduser("~/figure1_table.csv"), index=False)
print(f"Table saved → ~/figure1_table.csv  ({len(summary)} providers)")


,rank,Provider,Search Type,Ad Appearances,Organic Appearances,Mean Ad Position,Mean Organic Position,Total Appearances
0,1,Care Agency In Durham,Home Care,0.000000,18.000000,—,5.8,18.000000
1,2,Care Homes in Chester,Care Homes,0.000000,16.000000,—,5.8,16.000000
2,3,Help at home from a paid carer,Home Care,0.000000,13.000000,—,6.2,13.000000
3,4,Home Care in Durham,Home Care,0.000000,13.000000,—,5.8,13.000000
4,5,Cost of Home Care in London,Home Care,0.000000,13.000000,—,7.3,13.000000
5,6,My Homecare Durham,Home Care,0.000000,11.000000,—,5.2,11.000000
6,7,Care at Home & Homecare Providers in Durham Area,Home Care,0.000000,10.000000,—,3.6,10.000000
7,8,Langley Park: Residential & Dementia care home in Durham,Care Homes,0.000000,9.000000,—,5.8,9.000000
8,9,Find A Provider Near You,Home Care,9.000000,0.000000,1.8,—,9.000000
9,10,Home Care & Support Services,Home Care,0.000000,8.000000,—,7.4,8.000000


Table saved → ~/figure1_table.csv  (226 providers)


# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CELL 8 — Visualisations  (Figure 1 — interactive dropdown)                 │
# └─────────────────────────────────────────────────────────────────────────────┘

In [14]:
# ============================================================
# CELL 8 — Visualisation Suite  (Figure 1 — interactive dropdown)
# ============================================================

AD_COL  = "#E74C3C"
ORG_COL = "#2980B9"
LP_COL  = "#27AE60"


def fig_bar_stacked(df: pd.DataFrame):
    """Stacked bar: top 25 providers by total appearances, split Ad / Organic."""
    grp = (
        df[df["result_type"].isin(["Ad", "Organic"])]
        .groupby(["provider_name", "result_type"])["postcode_sector"]
        .count().reset_index(name="n")
    )
    top25 = grp.groupby("provider_name")["n"].sum().nlargest(25).index
    grp   = grp[grp["provider_name"].isin(top25)]
    order = grp.groupby("provider_name")["n"].sum().sort_values(ascending=True).index.tolist()
    grp["provider_name"] = pd.Categorical(grp["provider_name"], categories=order, ordered=True)
    fig = px.bar(
        grp.sort_values("provider_name"),
        y="provider_name", x="n", color="result_type", orientation="h",
        color_discrete_map={"Ad": AD_COL, "Organic": ORG_COL},
        title="Figure 1a — Top 25 Providers: Appearances by Result Type",
        labels={"n": "Appearances", "provider_name": "Provider", "result_type": "Result Type"},
        template="plotly_white",
    )
    fig.update_layout(barmode="stack", height=650,
                      yaxis=dict(tickfont=dict(size=10)),
                      legend_title="Result Type", font=dict(family="Arial"),
                      title_font_size=14)
    return fig


def fig_bubble(df: pd.DataFrame):
    """Bubble: x=Organic appearances, y=Ad appearances, size=total, colour=search type."""
    prov = (
        df[df["result_type"].isin(["Ad", "Organic"])]
        .groupby(["provider_name", "search_type", "result_type"])["postcode_sector"]
        .count().reset_index(name="n")
        .pivot_table(index=["provider_name", "search_type"],
                     columns="result_type", values="n", fill_value=0)
        .reset_index()
    )
    for c in ["Ad", "Organic"]:
        if c not in prov.columns:
            prov[c] = 0
    prov["total"] = prov["Ad"] + prov["Organic"]
    prov = prov[prov["total"] > 0].nlargest(40, "total")
    fig = px.scatter(
        prov, x="Organic", y="Ad", size="total", color="search_type",
        hover_name="provider_name",
        color_discrete_map={"Home Care": ORG_COL, "Care Homes": "#8E44AD"},
        size_max=55,
        title="Figure 1b — Paid vs Organic Visibility (bubble size = total appearances)",
        labels={"Organic": "Organic appearances", "Ad": "Ad appearances"},
        template="plotly_white",
    )
    fig.update_layout(legend_title="Search Type", font=dict(family="Arial"), title_font_size=14)
    return fig


def fig_heatmap(df: pd.DataFrame):
    """Heatmap: districts × provider (top 15), cell = appearances."""
    top15 = (
        df[df["result_type"].isin(["Ad", "Organic"])]
        .groupby("provider_name")["postcode_sector"]
        .count().nlargest(15).index
    )
    heat = (
        df[df["provider_name"].isin(top15)]
        .groupby(["district", "provider_name"])["postcode_sector"]
        .count().reset_index(name="n")
        .pivot(index="district", columns="provider_name", values="n")
        .fillna(0)
    )
    fig = px.imshow(
        heat, aspect="auto", color_continuous_scale="YlOrRd",
        title="Figure 1c — Provider Appearance Heatmap by Durham District",
        labels=dict(color="Appearances"), template="plotly_white",
    )
    fig.update_layout(font=dict(family="Arial"), title_font_size=14,
                      xaxis=dict(tickangle=-40, tickfont=dict(size=9)))
    return fig


def fig_position_violin(df: pd.DataFrame):
    """Violin: search position distribution for top 15 providers."""
    top15 = (
        df[df["result_type"].isin(["Ad", "Organic"])]
        .groupby("provider_name")["postcode_sector"]
        .count().nlargest(15).index
    )
    sub = df[df["provider_name"].isin(top15) & df["position"].notna()
             & df["result_type"].isin(["Ad", "Organic"])]
    fig = px.violin(
        sub, x="provider_name", y="position", color="result_type",
        box=True, points="all",
        color_discrete_map={"Ad": AD_COL, "Organic": ORG_COL},
        title="Figure 1d — Search Position Distribution for Top 15 Providers",
        labels={"position": "Search position (1 = top)", "provider_name": "Provider"},
        template="plotly_white",
    )
    fig.update_layout(yaxis_autorange="reversed",
                      xaxis=dict(tickangle=-40, tickfont=dict(size=9)),
                      font=dict(family="Arial"), title_font_size=14, height=540,
                      legend_title="Result Type")
    return fig


def fig_searchtype_split(df: pd.DataFrame):
    """Split bar: top 20 providers by Home Care vs Care Homes appearances."""
    grp = (
        df[df["result_type"].isin(["Ad", "Organic"])]
        .groupby(["provider_name", "search_type"])["postcode_sector"]
        .count().reset_index(name="n")
    )
    top20 = grp.groupby("provider_name")["n"].sum().nlargest(20).index
    grp   = grp[grp["provider_name"].isin(top20)]
    order = grp.groupby("provider_name")["n"].sum().sort_values(ascending=True).index.tolist()
    grp["provider_name"] = pd.Categorical(grp["provider_name"], categories=order, ordered=True)
    fig = px.bar(
        grp.sort_values("provider_name"),
        y="provider_name", x="n", color="search_type", orientation="h",
        color_discrete_map={"Home Care": ORG_COL, "Care Homes": "#8E44AD"},
        title="Figure 1e — Top 20 Providers: Home Care vs Care Homes Appearances",
        labels={"n": "Appearances", "provider_name": "Provider", "search_type": "Search Type"},
        template="plotly_white",
    )
    fig.update_layout(barmode="stack", height=600,
                      yaxis=dict(tickfont=dict(size=10)),
                      font=dict(family="Arial"), title_font_size=14,
                      legend_title="Search Type")
    return fig


FIGURES = {
    "1a — Stacked Bar: Ad vs Organic by Provider": fig_bar_stacked,
    "1b — Bubble: Paid vs Organic visibility":     fig_bubble,
    "1c — Heatmap: Provider × District":           fig_heatmap,
    "1d — Violin: Search position distribution":   fig_position_violin,
    "1e — Split Bar: Home Care vs Care Homes":     fig_searchtype_split,
}

dropdown = widgets.Dropdown(
    options=list(FIGURES.keys()),
    value=list(FIGURES.keys())[0],
    description="Figure 1:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="60%"),
)
out = widgets.Output()

def on_select(change):
    if change["type"] == "change" and change["name"] == "value":
        with out:
            clear_output(wait=True)
            FIGURES[change["new"]](df).show()

dropdown.observe(on_select)
display(dropdown, out)
with out:
    fig_bar_stacked(df).show()


Dropdown(description='Figure 1:', layout=Layout(width='60%'), options=('1a — Stacked Bar: Ad vs Organic by Pro…

Output()

# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CELL 9 — Export static Figure 1 for publication                             │
# │          pip install kaleido  (if not already installed)                    │
# └─────────────────────────────────────────────────────────────────────────────┘

In [1]:
# ============================================================
# CELL 9 — Export static Figure 1 for publication
# pip install kaleido  (if not already installed)
# ============================================================

import plotly.io as pio

export_fig = fig_bar_stacked(df)   # swap to any of the figure functions above

out_png  = os.path.expanduser("~/figure1.png")
out_html = os.path.expanduser("~/figure1_interactive.html")

pio.write_image(export_fig, out_png,  width=1400, height=750, scale=2)
pio.write_html(export_fig,  out_html)

print(f"✅ Exported:")
print(f"   Static PNG   → {out_png}")
print(f"   Interactive  → {out_html}")


NameError: name 'fig_bar_stacked' is not defined